# PERSEUS Rerun — CaRB Evaluation (TEMPLATE, heavily redacted)

End-to-end pipeline: Llama-3 extraction → repair → 5-stage validation → evaluation → stats.

> This template shows only cell structure and pipeline stages. Prompts, thresholds, algorithm bodies, and logging/IO details are redacted.

**Workflow the notebook implements:**
1. Load CaRB GT and sentences.
2. Run Llama-3 extraction over every sentence. Parse bullet-list triples.
3. Apply R1 (canonicalization) + R2 (grounding) repair.
4. Five-stage validation: lexical check → local NLI → window NLI → BM25 retrieval → retrieval NLI.
5. Evaluate PERSEUS and Raw LLaMA baseline with 3-tier SBERT matching.
6. McNemar test + bootstrap CIs for precision / recall / F1.
7. Error analysis, timing, final summary.

## Cell 0 — Configuration

In [ ]:
# Imports, paths, model names, thresholds, vocab, logging setup.

import os, sys, json, re, math, time, warnings
from datetime import datetime
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

SMOKE_TEST = True   # 5-sentence dry run; set False for full 775-sentence run

# ---- Paths -----------------------------------------------------------
PROJECT_ROOT = Path(r'<REDACTED>')
DATA_DIR     = PROJECT_ROOT / 'data'
OUTPUTS_DIR  = PROJECT_ROOT / 'outputs'
LOGS_DIR     = PROJECT_ROOT / 'logs'
GT_PATH      = DATA_DIR / 'carb_wiki.csv'

RUN_ID  = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUTS_DIR / f'perseus_rerun_{RUN_ID}'
# <REDACTED: mkdirs, log path, Tee-to-file stdout wrapper>

# ---- Models ----------------------------------------------------------
HF_TOKEN        = '<REDACTED>'
LLM_MODEL_NAME  = 'meta-llama/Meta-Llama-3-8B-Instruct'
NLI_MODEL_NAME  = 'roberta-large-mnli'
SBERT_MODEL     = 'all-mpnet-base-v2'
SPACY_MODEL     = 'en_core_web_sm'

# ---- Extraction config ----------------------------------------------
EXTRACTION_TEMPERATURE = 0.0
EXTRACTION_MAX_TOKENS  = 600
EXTRACTION_DO_SAMPLE   = False
EXTRACTION_SYSTEM_PROMPT = '<REDACTED>'
EXTRACTION_USER_TEMPLATE = '<REDACTED: takes {sentence} placeholder>'

# ---- Thresholds (all redacted; see paper for values) ----------------
# Repair span scoring, NLI decision margins, retrieval trigger,
# BM25, tier-2/3 match, bootstrap config.
# <REDACTED>

# ---- Vocabulary sets (all redacted) ---------------------------------
# STOPWORDS, NEGATION_CUES, MODALITY_CUES, COPULAS, PLACEHOLDERS, PRONOUNS
# <REDACTED>

# ---- Save 00_run_config.json + 00_environment.txt -------------------
# <REDACTED: dump run_config dict and pip freeze + GPU info>


## Cell 1 — Load models

Llama-3-8B-Instruct, RoBERTa-Large-MNLI, SBERT (all-mpnet-base-v2), spaCy. ~2–3 min first load.

In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM, pipeline,
                          AutoModelForSequenceClassification)
from sentence_transformers import SentenceTransformer, util
import spacy

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# <REDACTED: load llama_tokenizer, llama_model (bf16, device_map='auto'),
#  build llama_pipeline>
llama_tokenizer = None
llama_model     = None
llama_pipeline  = None

# <REDACTED: load nli_tokenizer, nli_model (on DEVICE), cache nli_id2label>
nli_tokenizer = None
nli_model     = None
nli_id2label  = None

# <REDACTED: load sbert_model and spaCy nlp; auto-download spaCy if missing>
sbert_model = None
nlp         = None


## Cell 2 — Load CaRB and normalize ground truth

In [ ]:
# Read GT csv, standardize column names to Subject/Predicate/Object/sentence,
# build per-sentence table with 'sent_XXXX' ids, restrict to 5 sentences
# if SMOKE_TEST.

gt_df              = pd.read_csv(GT_PATH)
# <REDACTED: column renaming via lowercase map>
carb_sentences_df  = None   # built from unique gt_df['sentence']
SENTENCES          = None   # sent_id -> sentence dict
ALL_SENT_IDS       = None   # ordered list of sent_ids

def norm_text(s):       """Lowercase, strip non-alphanumeric, collapse whitespace."""; pass
def canon_entity(s):    """norm_text minus stopwords."""; pass
def norm_pred(p):       """Strip leading/trailing copulas from predicate."""; pass
def normalize_triples_df(df):
    """Add Snorm/Pnorm/Onorm columns, drop rows where any is empty."""
    pass

# <REDACTED: if SMOKE_TEST, filter gt_df to matching sentences>

gt_normalized = normalize_triples_df(gt_df)
gt_unique     = None   # drop_duplicates on [Snorm, Pnorm, Onorm]

# <REDACTED: save 01_carb_sentences.csv, 01_carb_ground_truth.csv, 01_carb_ground_truth_normalized.csv>


## Cell 3 — Llama-3 extraction

For each CaRB sentence: chat-template prompt → Llama generation → bullet-list parse → per-sentence dedup. Saves raw responses, parsed triples, malformed lines, and stats.

In [ ]:
def llama_extract(sentence):
    """Build chat messages from EXTRACTION_SYSTEM_PROMPT + EXTRACTION_USER_TEMPLATE,
    apply chat template, run llama_pipeline, return stripped generated text."""
    pass

def parse_triples(raw_output):
    """Parse bullet-list triples from Llama output.
    Filters placeholders, merges >3-part splits, rejects too-short predicates.
    Returns (triples_list, malformed_lines_list)."""
    pass

# ---- Main extraction loop -------------------------------------------
# for each sent_id:
#   raw = llama_extract(sentence)   [wrapped in try/except for robustness]
#   triples, malformed = parse_triples(raw)
#   accumulate raw_records, parsed_triples, dropped_malformed
#   print progress every 25 sentences with avg time / ETA
# <REDACTED: loop body>

# ---- Post-extraction -----------------------------------------------
# <REDACTED: save 02_extraction_raw_responses.jsonl,
#  02_extraction_parsed_triples.csv, _dedup.csv, _dropped_malformed.jsonl,
#  02_extraction_stats.json>

RAW_LLAMA_TRIPLES = None    # deduped parsed triples; this is the baseline


## Cell 4 — Triple repair (R1 + R2)

R1: canonicalize subject/object spans against the sentence, normalize predicate, preserve negation/modality cues.
R2: verify all content tokens are grounded in the sentence.

In [ ]:
PRED_NORM_MAP = {'<REDACTED: surface -> canonical predicate map>': '...'}

def best_span(query, sentence, min_score, token_w, seq_w):
    """Scan contiguous spans near query length, score by weighted combination
    of token-overlap and SequenceMatcher ratio, return best >= min_score."""
    pass

def repair_triple(subject, predicate, obj, sentence):
    """R1 + R2 repair.

    Returns dict with keys:
        status   : 'OK' | 'DROP'
        subject_r, predicate_r, object_r : repaired strings (None if DROP)
        edits    : list of edit descriptions
        drop_reason : string if DROP
        scores   : {'subject_span_score', 'object_span_score'} if used

    Steps: ground subject via best_span -> ground object via best_span ->
    normalize predicate -> preserve negation cue -> preserve modality cue ->
    content-token grounding check -> final negation/modality check.
    """
    pass

# ---- Apply to all raw triples ---------------------------------------
# for each row in RAW_LLAMA_TRIPLES: run repair_triple, tally edits and drops,
# split into OK and DROPPED lists.
# <REDACTED: loop body>

# <REDACTED: save 03_repair_output.jsonl, 03_repair_OK.csv,
#  03_repair_DROPPED.csv, 03_repair_stats.json>

REPAIRED_OK = None   # DataFrame of accepted repaired triples


## Cell 5 — Validation pipeline

Stages: (5a) lexical check · (5b) local + window NLI · (5c) retrieval trigger policy · (5d) BM25 retrieval · (5e) retrieval NLI · (5f) final tagging with error types.

In [ ]:
def lex_score(subject, obj, sentence):
    """Return (score in {0,1,2}, subj_hit bool, obj_hit bool)."""
    pass

def get_window(sent_id, radius=1):
    """Concatenate +/- radius neighboring CaRB sentences."""
    pass

def triple_to_hyp(s, p, o):
    """Return the NLI hypothesis string from a triple."""
    pass

def nli_scores(premise, hypothesis):
    """Run nli_model, return {'entail', 'neutral', 'contra'} floats."""
    pass

def decide_nli(scores):
    """Return 'ACCEPT' | 'REJECT' | 'UNVERIFIED' based on entail/contra
    floors and margins against neutral."""
    pass

def tokenize_bm25(s):       """BM25 tokenizer (lowercase, split on non-word)."""; pass
def bm25_score_doc(q, idx): """Okapi BM25 against doc_tokens[idx]."""; pass
def retrieve_topk(query, top_k, exclude_sent_id=None):
    """Rank ALL_SENT_IDS by BM25, return top_k as [(sent_id, score), ...]."""
    pass

def infer_error_type(rec, final):
    """Categorize rejects: coreference_required | negation_mismatch |
    contradiction_or_false | low_support."""
    pass

# ---- Stage 5a: lexical check ----------------------------------------
# Tag each repaired triple as LOCAL_ACCEPT / UNCERTAIN / LOCAL_REJECT
# based on lex_score == 2 / 1 / 0. Save 04a_lexical_check.jsonl.
# <REDACTED>

# ---- Stage 5b: local NLI + window escalation ------------------------
# For non-LOCAL_REJECT: run NLI(sentence, hyp). If UNVERIFIED, retry with
# a +/-1 sentence window. Track NLI call count. Save 04b / 04c jsonl.
# <REDACTED>

# ---- Stage 5c: retrieval trigger policy -----------------------------
# Trigger if UNVERIFIED, or if ACCEPT with weak lex (<WEAK_LEX_THRESHOLD)
# or weak entail (<WEAK_NLI_THRESHOLD). Never trigger on REJECT.
# Save 04d_trigger_policy.jsonl.
# <REDACTED>

# ---- Stage 5d: BM25 retrieval ---------------------------------------
# Build corpus over ALL_SENT_IDS, retrieve_topk for every triggered triple,
# excluding its source sentence. Save 04e_bm25_retrieval.jsonl.
# <REDACTED>

# ---- Stage 5e: retrieval-stage NLI ----------------------------------
# For each triggered triple, run NLI against each hit in order; early-exit
# on first ACCEPT or REJECT. Save 04f_retrieval_nli.jsonl.
# <REDACTED>

# ---- Stage 5f: final tagging + error types --------------------------
# Freeze final_decision_final per triple and assign error_type to rejects.
# Save 04g_final_tagged.jsonl and 04_validation_stats.json (incl. NLI calls saved %).
# <REDACTED>

stage_f_records   = None
validation_stats  = None


## Cell 6 — PERSEUS predictions

In [ ]:
# Split stage_f_records by final_decision_final into ACCEPT / REJECT / UNVERIFIED
# Save 05_perseus_ACCEPT.csv, 05_perseus_REJECT.csv, 05_perseus_UNVERIFIED.csv.
# <REDACTED>

PERSEUS_PREDS = None   # DataFrame of accepted predictions
accept_rows   = None   # list backing PERSEUS_PREDS (reused by error analysis)


## Cell 7 — Evaluation (3-tier SBERT matching)

Runs the same matcher against PERSEUS and Raw LLaMA. Tier 1: exact. Tier 2: SBERT cosine ≥ TIER2_THRESHOLD. Tier 3: cosine ≥ TIER3_THRESHOLD. Greedy 1-1 assignment.

In [ ]:
def evaluate_system(pred_df, gt_df, system_name, save_prefix):
    """3-tier SBERT evaluation with arg-swap tolerance.

    Normalizes preds, builds tstr and tstr_sw forms, tries exact match first,
    then SBERT cosine (max over both forms) with greedy descending-similarity
    assignment. Emits TP / FP / FN CSVs and a metrics JSON.

    Returns (metrics_dict, gt_matched_set, matches_df).
    """
    pass

perseus_metrics,  perseus_gt_matched,  perseus_matches_df  = evaluate_system(
    PERSEUS_PREDS,     gt_unique, 'PERSEUS',   '06_perseus')

baseline_metrics, baseline_gt_matched, baseline_matches_df = evaluate_system(
    RAW_LLAMA_TRIPLES, gt_unique, 'Raw LLaMA', '06_baseline')

# <REDACTED: build and save 06_system_comparison.csv>


## Cell 8 — Statistical tests (McNemar + bootstrap CIs)

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test

TOTAL_GT = len(gt_unique)

# ---- McNemar contingency (a=both, b=perseus only, c=baseline only, d=neither)
# Uses exact binomial when (b+c) < 25, else chi-squared with Yates correction.
# <REDACTED: table construction, mcnemar_test call, save 07_mcnemar_test.json>
# <REDACTED: save 07_mcnemar_discordant_{perseus,baseline}_only.csv>

# ---- Bootstrap CIs (percentile, BOOTSTRAP_N_RESAMPLES resamples) ----
def bootstrap_metric(pred_labels, gt_labels, metric_func, n_resamples):
    """Independent-resample percentile bootstrap for the given metric."""
    pass

def precision_fn(pl, gl):  """TP / n_pred"""; pass
def recall_fn(pl, gl):     """TP / n_gt"""; pass
def f1_fn(pl, gl):         """harmonic mean"""; pass

def build_labels(matches_df, gt_matched_set, n_pred, n_gt):
    """Per-pred TP/FP labels + per-GT matched/unmatched labels."""
    pass

# <REDACTED: run bootstrap for PERSEUS (P/R/F1) and Baseline (P/R/F1).
#  Save 07_bootstrap_{perseus,baseline}_{precision,recall,f1}_CI.json>

# ---- Delta CIs (PERSEUS - Baseline, independent-resample approximation)
# <REDACTED: compute delta arrays from resample samples, quantile CIs,
#  save 07_bootstrap_delta_{precision,f1}_CI.json>


## Cells 9–11 — Error analysis, timing, summary

In [ ]:
# ===== Cell 9: Error analysis =====
# - 08_tier_distribution.csv          (count + % by tier1_exact / tier2_semantic / tier3_fuzzy / no_match)
# - 08_error_type_breakdown.csv       (counts of each error_type across stage_f_records)
# - 08_negation_analysis.csv          (sentences with negation cues, whether predicate preserved it)
# - 08_repair_impact.csv              (per edit_type, how often the triple was ultimately accepted)
# - 08_per_sentence_metrics.csv       (per sent_id: #preds, TP, FP)
# - 08_per_predicate_metrics.csv      (per predicate: #preds, TP, precision)
# <REDACTED>

# ===== Cell 10: Timing + triple flow =====
# - 09_stage_timing.json              (extraction/repair totals, avg per sentence, NLI calls saved)
# - 09_triple_flow.csv                (per-triple trace through stages)
# - 09_nli_calls_saved.json           (actual vs max NLI calls, % saved)
# <REDACTED>

# ===== Cell 11: Final summary =====
# - 10_FINAL_SUMMARY.txt              (PERSEUS / baseline P/R/F1 with CIs, deltas, McNemar)
# - 10_comparison_with_paper.csv      (reference-value diff table; redacted values)
# <REDACTED>

# Close log file, restore stdout, print completion banner.
# <REDACTED>
